# xHuBERT Experiment 3: xHuBERT Fine-tuning (Main Result)
**De tai**: He thong goi y san pham dua tren phan tich giong noi va cam xuc
**Hoc vien**: Nguyen Tan Nhu | **GVHD**: TS. Bui Thanh Hung (IUH)

**Yeu cau**: `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save

## Pipeline
```
Step 0: Setup
Step 1: xHuBERT fine-tune: 5-fold CV (seed 42) + LOSGO (seeds 42,43,44)
Step 2: Extract embeddings for Exp5
Step 3: Visualization
```

**Estimated time**: ~7h (seed 42) + ~12h (seeds 43,44)

In [ ]:
# Kiem tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")
else:
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/xhubert_results/"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

In [ ]:
# Install dependencies (Colab has torch, numpy, sklearn, matplotlib)
!pip install -q transformers==4.51.3 librosa huggingface-hub safetensors tqdm seaborn

# Verify versions
import torch, transformers, librosa
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"librosa:      {librosa.__version__}")

In [ ]:
# Upload .py modules to Colab
# Option 1: Upload manually via Files panel (drag & drop)
# Option 2: Clone from repo
# !git clone https://github.com/nhunet/xhubert-experiments.git
# %cd xhubert-experiments

# Verify required files
import os
required = [
    "config.py", "data.py", "features.py", "protocols.py",
    "stats.py", "utils.py",
    "models/__init__.py", "models/ml_classifiers.py",
    "models/xhubert.py", "models/hubert_vanilla.py",
    "models/fusion.py", "models/dl_1d.py", "models/dl_2d.py",
]
for f in required:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}]  {f}")

In [ ]:
# Download RAVDESS dataset
import os
RAVDESS_PATH = "./RAVDESS"
if not os.path.exists(RAVDESS_PATH):
    print("Downloading RAVDESS ...")
    !wget -q https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
    !unzip -q Audio_Speech_Actors_01-24.zip -d RAVDESS/
    print("Done!")
else:
    print(f"RAVDESS already exists at {RAVDESS_PATH}")
    !find {RAVDESS_PATH} -name "*.wav" | wc -l

In [ ]:
# Set save directory
import config
config.SAVE_DIR = SAVE_DIR
config.CKPT_DIR = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(config.CKPT_DIR, exist_ok=True)
print(f"Results -> {config.SAVE_DIR}")
print(f"Checkpoints -> {config.CKPT_DIR}")

### Keep Colab Alive
Paste this into your **browser Console** (F12 -> Console) to prevent idle timeout:
```javascript
function ClickConnect() {
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

## Load Dataset

In [ ]:
from data import RavdessDataset
import config

dataset = RavdessDataset(sr=config.SR_HUBERT)
dataset.print_summary()

## Quick Test (1 fold, 2 epochs)

In [ ]:
from experiments.exp3_xhubert_finetune import run_exp3

# Quick test to verify pipeline
df_quick = run_exp3(dataset=dataset, seeds=[42], protocols=["LOSGO"],
                    force=True, quick=True)
print("Quick test passed!" if len(df_quick) > 0 else "FAILED!")

## Full Run: 5-fold CV (seed 42)

In [ ]:
# 5-fold CV with seed 42 (~2h)
df_5fold = run_exp3(dataset=dataset, seeds=[42], protocols=["5-fold CV"], force=False)
print(df_5fold.groupby("Protocol")["accuracy"].agg(["mean", "std"]).round(2))

## Full Run: LOSGO (seed 42)

In [ ]:
# LOSGO with seed 42 (~5h)
df_losgo_42 = run_exp3(dataset=dataset, seeds=[42], protocols=["LOSGO"], force=False)
print(df_losgo_42.groupby("Protocol")["accuracy"].agg(["mean", "std"]).round(2))

## LOSGO seeds 43, 44 (run in Session 2 if timeout)

In [ ]:
# LOSGO seeds 43, 44 (~12h total)
df_losgo_extra = run_exp3(dataset=dataset, seeds=[43, 44],
                          protocols=["LOSGO"], force=False)
print(df_losgo_extra.groupby(["Protocol", "Seed"])["accuracy"].agg(["mean", "std"]).round(2))

## Visualization

In [ ]:
from visualization.plots import plot_exp3_comparison, plot_confusion_matrix
import pandas as pd, os, numpy as np

# Load all Exp3 results
exp3_csv = os.path.join(config.SAVE_DIR, "results_exp3_xhubert.csv")
df_exp3 = pd.read_csv(exp3_csv) if os.path.exists(exp3_csv) else df_5fold

# Load Exp2 for comparison
exp2_csv = os.path.join(config.SAVE_DIR, "results_exp2_hubert_frozen.csv")
df_exp2 = pd.read_csv(exp2_csv) if os.path.exists(exp2_csv) else None

plot_exp3_comparison(df_exp3, df_exp2)
print("Figures saved!")

## Summary

In [ ]:
import pandas as pd, os

exp3_csv = os.path.join(config.SAVE_DIR, "results_exp3_xhubert.csv")
if os.path.exists(exp3_csv):
    df = pd.read_csv(exp3_csv)
    print("=== xHuBERT Results ===")
    print(df.groupby(["Protocol", "Seed"])[["accuracy", "f1_macro"]].agg(["mean", "std"]).round(2))
else:
    print("Run experiments first!")